In [2]:
from datetime import date
import requests
import pandas as pd
import time
pd.set_option('display.max_columns', None)


# Setup

In [1]:
institution_ror = catalog.load('params:openalex_extract_options.institution_ror')
author_filter = catalog.load('params:openalex_extract_options.author_filter')
institution_filter = catalog.load('params:openalex_extract_options.institution_filter')
work_filter = catalog.load('params:openalex_extract_options.work_filter')

#env = catalog.load('params:fetch_options.env')
env = 'dev'

print(f'institution_ror: {institution_ror}')
print(f'author_filter: {author_filter}')
print(f'institution_filter: {institution_filter}')
print(f'work_filter: {work_filter}')
print(f'env: {env}')


[12/15/25 11:27:42] INFO     Loading data from params:openalex_extract_options.institution_ror ]8;id=439235;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=515707;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             (MemoryDataset)...                                                                    

                    INFO     Loading data from params:openalex_extract_options.author_filter   ]8;id=204229;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=506089;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             (MemoryDataset)...                                                                    

                    INFO     Loading data from                                                 ]8;id=247026;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=142471;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             params:openalex_extract_options.institution_filter                                    
                             (MemoryDataset)...                                                                    

                    INFO     Loading data from params:openalex_extract_options.work_filter     ]8;id=526482;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=989769;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             (MemoryDataset)...                                                                    

institution_ror: https://ror.org/02s7sax82
author_filter: affiliations.institution.ror
institution_filter: ror
work_filter: institutions.ror
env: dev


In [ ]:
institution_ror = 'https://ror.org/01tjs6929'

# Node

In [ ]:
def openalex_extract(institution_ror: str, filter_field: str, entity: str = 'institutions', env: str = 'dev', cleaner=None):
    """
    Fetch data from OpenAlex API for a given entity and institution ROR.

    Args:
        entity (str): 'authors', 'institutions', 'works', etc.
        institution_ror (str): ROR id of the institution.
        env (str): 'dev' or 'prod'.
        filter_field (str): the filter key to use (e.g. 'affiliations.institution.ror').
        cleaner (callable): function to clean DataFrame columns, optional.

    Returns:
        pd.DataFrame: full concatenated results
        pd.DataFrame: head(1000) sample
    """
    session = requests.Session()
    base_url = f"https://api.openalex.org/{entity}?filter={filter_field}:{{}}&cursor={{}}&per-page=200"
    cursor = '*'
    iteration_limit = 5
    iteration_count = 0
    all_dataframes = []

    while True:
        url = base_url.format(institution_ror, cursor)
        print(f'Iteration count: {iteration_count}')
        print(f'GET {url}')

        try:
            response = session.get(url, timeout=10)
            response.raise_for_status()
            api_response = response.json()
        except requests.RequestException as e:
            print(f"Error en la solicitud: {e}")
            break
        except ValueError:
            print("Error al decodificar JSON.")
            break

        if 'results' not in api_response or not api_response['results']:
            print("No hay más datos disponibles.")
            break

        df_tmp = pd.DataFrame.from_dict(api_response['results'])

        columns_to_drop = {"abstract_inverted_index", "abstract_inverted_index_v3"}
        df_tmp = df_tmp.drop(columns=columns_to_drop.intersection(df_tmp.columns))

        all_dataframes.append(df_tmp)

        # update cursor
        cursor = api_response.get('meta', {}).get('next_cursor')
        if not cursor:
            break

        iteration_count += 1
        if env == 'dev' and iteration_count >= iteration_limit:
            break

        time.sleep(1)

    df = pd.concat(all_dataframes, ignore_index=True) if all_dataframes else pd.DataFrame()


    df['_filter_param'] = filter_field
    df['_filter_value'] = institution_ror
    df['_extract_datetime'] = date.today()

    return df, df.head(1000)


# Results

In [ ]:
df_institution, df_institution_dev = openalex_extract(institution_ror, institution_filter)

In [ ]:
df_institution_dev

In [ ]:
df_author, df_author_dev = openalex_extract(institution_ror, author_filter, 'authors')

In [ ]:
df_author_dev

In [ ]:
df_work, df_work_dev = openalex_extract(institution_ror, work_filter, 'works')

In [ ]:
df_work_dev